### **Step 1:** Look at the observation data.

In [21]:
import pandas
import duckdb

connection = duckdb.connect('novellia.duckdb')
connection.execute('''
SELECT
    *
FROM
    stg_observation
''').df()

,observation_id,status,category,observation_name,observation_code,observation_code_system,patient_id,encounter_id,effective_date_time,issued,value_quantity,value_unit,value_codeable_concept,value_string
0,0000019b-1ad3-f95b-90f7-7162d4dfb4bd,final,laboratory,Ketones [Presence] in Urine by Test strip,2514-8,http://loinc.org,4109755f-085a-5908-abdf-168596406e33,2d937df2-28e2-0521-f970-8ae48dbfc995,1985-02-13T02:50:00-05:00,1985-02-13T02:50:00.783-05:00,NaN,None,Urine ketone test = +++ (finding),None
1,00000c76-af56-c3c3-972a-8d3a2b4723e1,final,laboratory,"Carbon dioxide, total [Moles/volume] in Blood",20565-8,http://loinc.org,93b485d5-921c-f94c-526e-99a615416965,4d806e90-2e17-6a09-be1e-cda5eda5612b,2015-02-27T14:43:29-05:00,2015-02-27T14:43:29.368-05:00,26.520,mmol/L,None,None
2,00001a82-0312-fe4c-7391-b5cac9f2007c,final,vital-signs,Body Height,8302-2,http://loinc.org,e33cbac7-06e7-425a-e3b5-c23db2d8e9ff,de69024c-0fc1-2990-3c67-1734bcbec5dd,2015-08-01T10:34:35-04:00,2015-08-01T10:34:35.060-04:00,67.100,cm,None,None
3,0000241b-2c9e-389c-1b3a-08afeca79fbf,final,laboratory,Aspartate aminotransferase [Enzymatic activity...,1920-8,http://loinc.org,ad239efc-637c-e428-c829-b87e700d5446,6e74946d-8ed9-466f-6739-5802f2260848,2018-03-15T11:47:06-04:00,2018-03-15T11:47:06.955-04:00,37.191,U/L,None,None
4,00004e10-6835-8173-f786-42aef417abaa,final,laboratory,Potassium [Moles/volume] in Blood,6298-4,http://loinc.org,d8f56f83-4f2a-0133-8335-e053343e5766,0294d1a2-254b-5a6f-77b0-a6bcfe21e5e3,2004-01-09T13:36:59-05:00,2004-01-09T13:36:59.676-05:00,4.980,mmol/L,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
693518,ffffa437-9c20-0d27-a09b-12eec3eed39f,final,laboratory,Color of Urine,5778-6,http://loinc.org,9357bf54-856c-2858-8a79-df6e00ff31bc,10582222-fd67-e5db-36f6-cfcc612e150a,2002-04-27T23:59:51-04:00,2002-04-27T23:59:51.812-04:00,NaN,None,Brown color (qualifier value),None
693519,ffffcb6a-2b7f-1503-ada9-fff16ba523f4,final,laboratory,"Carbon dioxide, total [Moles/volume] in Blood",20565-8,http://loinc.org,7d9aa431-cd72-8aa2-9559-5920937d9330,f36c1700-b072-3439-737e-41dc4a12193d,2005-11-24T07:05:37-05:00,2005-11-24T07:05:37.085-05:00,25.240,mmol/L,None,None
693520,ffffdcbb-5f60-c630-98d3-37fe21cff3de,final,vital-signs,Heart rate,8867-4,http://loinc.org,ac91b90d-97e4-4fc5-41cd-036bac49e6e8,85daa6fc-1471-931f-fa1d-05774ce26832,2016-06-25T02:21:46-04:00,2016-06-25T02:21:46.792-04:00,68.000,/min,None,None
693521,ffffe7d8-2356-a197-7762-737d1a6bbce1,final,laboratory,Chloride [Moles/volume] in Serum or Plasma,2075-0,http://loinc.org,ad239efc-637c-e428-c829-b87e700d5446,6373f310-7ca7-1871-9158-75d86b4b62c3,2013-10-18T12:35:06-04:00,2013-10-18T12:35:06.955-04:00,108.390,mmol/L,None,None


### **Step 2:** Look at values that could filter down the results.

In [22]:
connection.execute('''
SELECT
    DISTINCT status,
    CASE
        WHEN issued IS NOT NULL THEN 'include'
        ELSE 'exclude'
    END AS include_exclude
FROM
    stg_observation
''').df()

,status,include_exclude
0,final,include


### **Step 3:** Look at/validate all possible categories. We're good with filtering for just vital-signs.

In [23]:
connection.execute('''
SELECT
    DISTINCT category
FROM
    stg_observation
''').df()

,category
0,imaging
1,social-history
2,laboratory
3,survey
4,therapy
5,vital-signs
6,exam
7,procedure


### **Step 4:** Pre-aggregate and count the # of vital signs observations for every patient, and then calculate the average across the entire population. Validate the # of vital signs observations in the dataset cuz 146 per patient seems crazy high at first glance.

**Bonus:** I also wanted to understand - what's the relationship between an observation and an encounter? Looks like an encounter can have many observations, but it's limited only to ever 1 patient. This makes sense - an encounter is a patient-specific happening, that can contain many observations (i.e. getting multiple tests and/or types of tests in a single visit).

In [24]:
connection.execute('''
WITH per_patient_count AS (
    SELECT
        patient_id,
        COUNT(observation_id) AS vital_signs_measurements
    FROM
        stg_observation
    WHERE
        category = 'vital-signs'
    GROUP BY
        1
)
SELECT
    AVG(vital_signs_measurements)
FROM
    per_patient_count
''').df()

,avg(vital_signs_measurements)
0,146.716783


In [25]:
connection.execute('''
    SELECT
        COUNT(observation_id) FROM stg_observation WHERE category = 'vital-signs'
''').df()

,count(observation_id)
0,167844


In [26]:
connection.execute('''
SELECT
    encounter_id,
    COUNT(DISTINCT observation_id),
    COUNT(DISTINCT patient_id)
FROM
    stg_observation
GROUP BY
    1
ORDER BY 3 DESC
''').df()

,encounter_id,count(DISTINCT observation_id),count(DISTINCT patient_id)
0,41e482cc-218d-dc24-0a0e-0b3b8e519eb1,12,1
1,be024f62-f96f-9e44-1c0e-ebdba8752145,41,1
2,ca2a1886-7250-c8dd-8fbc-d0187159e3f6,33,1
3,fc259532-5cc1-777d-dbc3-f6a976e63488,47,1
4,3d4bbce4-02e5-e98a-17a4-3ab625bce335,38,1
...,...,...,...
46904,81d72971-2771-1d93-ff41-859e064f6040,1,1
46905,365a3c78-5425-103b-1fc0-5dc341935252,1,1
46906,26751621-c64f-1e59-6f7c-fcd4b139d9e5,1,1
46907,d88e8749-49cc-65b2-d104-ac41abecfa15,1,1


### **ANSWER 03**
On average, a given patient has 146 vital signs measurements/observations in this dataset.